# Example 3: A Different Genetic Code — Human Mitochondrial COX1


Fetches the human *MT-CO1* (cytochrome c oxidase subunit I) gene from the revised
Cambridge Reference Sequence (`NC_012920.1`, positions 5,904–7,445, 1,542 bp) and
translates the exact same sequence under both the standard genetic code (table 1) and the
vertebrate mitochondrial code (table 2).


The vertebrate mitochondrial code differs from the standard code at exactly
four codons — `AGA`/`AGG` (Arg → Stop), `ATA` (Ile → Met), and `TGA` (Stop → Trp).

In [1]:
# Fetch human MT-CO1 and translate it under both the standard and the
# vertebrate mitochondrial genetic code
import os
from Bio import Entrez, SeqIO
from Bio.Data import CodonTable

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

Entrez.email = "you@example.com"
handle = Entrez.efetch(db="nucleotide", id="NC_012920.1", rettype="fasta", retmode="text",
                        seq_start=5904, seq_stop=7445)
record = SeqIO.read(handle, "fasta")
handle.close()
SeqIO.write(record, "data/NC_012920.1_COX1.fasta", "fasta")
cds_seq = record.seq
print(f"COX1 CDS length: {len(cds_seq)} bp")

# Translate the SAME sequence under two different genetic code tables
standard_protein = cds_seq.translate(table=1, to_stop=True)
mito_protein = cds_seq.translate(table=2, to_stop=True)

print(f"Standard code (table 1):             {len(standard_protein)} aa before first stop")
print(f"Vertebrate mitochondrial (table 2):  {len(mito_protein)} aa before first stop")

# Count how often each of the four reassigned codons actually appears in this CDS
codons = [str(cds_seq[i:i+3]) for i in range(0, len(cds_seq) - 2, 3)]
standard_table = CodonTable.unambiguous_dna_by_id[1]
mito_table = CodonTable.unambiguous_dna_by_id[2]
reassigned = {"AGA": 0, "AGG": 0, "ATA": 0, "TGA": 0}
for codon in codons:
    if codon in reassigned:
        reassigned[codon] += 1

lines = [f"CDS length: {len(cds_seq)} bp ({len(codons)} codons)"]
lines.append(f"Standard-code translation before first stop: {len(standard_protein)} aa")
lines.append(f"Mitochondrial-code translation before first stop: {len(mito_protein)} aa")
lines.append("Occurrences of the four reassigned codons in this CDS:")
for codon, count in reassigned.items():
    std_aa = standard_table.forward_table.get(codon, "Stop")
    mito_aa = mito_table.forward_table.get(codon, "Stop")
    lines.append(f"  {codon}: {count} occurrence(s) — standard={std_aa}, mitochondrial={mito_aa}")

print("\n" + "\n".join(lines[-5:]))
with open("results/example3_cox1_code_comparison.txt", "w") as f:
    f.write("\n".join(lines) + "\n")
print("\nSaved: data/NC_012920.1_COX1.fasta, results/example3_cox1_code_comparison.txt")

COX1 CDS length: 1542 bp
Standard code (table 1):             5 aa before first stop
Vertebrate mitochondrial (table 2):  513 aa before first stop

Occurrences of the four reassigned codons in this CDS:
  AGA: 1 occurrence(s) — standard=R, mitochondrial=Stop
  AGG: 0 occurrence(s) — standard=R, mitochondrial=Stop
  ATA: 25 occurrence(s) — standard=I, mitochondrial=M
  TGA: 16 occurrence(s) — standard=Stop, mitochondrial=W

Saved: data/NC_012920.1_COX1.fasta, results/example3_cox1_code_comparison.txt
